# Hey Emma — Custom Wake Word Training

Trainiert ein openWakeWord-Modell für **"Hey Emma"** mit synthetischen Stimmen (Piper TTS).

**Voraussetzungen:** Google Colab mit GPU-Runtime (T4 reicht).

**Persistent Storage:** Alle Daten werden auf Google Drive gespeichert.
Bei Runtime-Reset einfach alle Zellen erneut ausführen — bereits vorhandene
Daten werden übersprungen.

**Ergebnis:** `hey_emma.onnx` in Google Drive unter `hey_emma_training/`.

## 1. Environment Setup

In [ ]:
# Mount Google Drive + set up persistent directory structure
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hey_emma_training'
DRIVE_DATA = f'{DRIVE_BASE}/training_data'
DRIVE_MODEL = f'{DRIVE_BASE}/hey_emma_model'

for d in [DRIVE_BASE, DRIVE_DATA, DRIVE_MODEL]:
    os.makedirs(d, exist_ok=True)

# Symlink Drive dirs to expected local paths
for local, remote in [('/content/training_data', DRIVE_DATA),
                      ('/content/hey_emma_model', DRIVE_MODEL)]:
    if os.path.islink(local):
        os.remove(local)
    if not os.path.exists(local):
        os.symlink(remote, local)
        print(f'Symlink: {local} -> {remote}')
    else:
        print(f'Exists: {local}')

os.chdir('/content')
print(f'Working directory: {os.getcwd()}')
print(f'All data persisted on Google Drive: {DRIVE_BASE}')

In [ ]:
# Install espeak-ng (required for Piper TTS phonemizer)
!apt-get install -y espeak-ng

# Install openWakeWord from GitHub (PyPI package doesn't include train.py)
!pip install git+https://github.com/dscripka/openWakeWord.git

# Training dependencies (not auto-installed, torchaudio is pre-installed on Colab)
!pip install onnx onnxruntime datasets huggingface_hub pyyaml \
    torchinfo torchmetrics mutagen audiomentations pronouncing \
    speechbrain torch-audiomentations acoustics soundfile torchcodec

In [ ]:
# Clone piper-sample-generator (must be local, not on Drive)
import os

if not os.path.exists('/content/piper-sample-generator/generate_samples.py'):
    !git clone https://github.com/dscripka/piper-sample-generator.git /content/piper-sample-generator
    !pip install -r /content/piper-sample-generator/requirements.txt
else:
    print('piper-sample-generator already cloned.')

# Download the Piper TTS model (check Drive cache first)
MODEL_NAME = 'en-us-libritts-high.pt'
DRIVE_PIPER_MODEL = f'/content/drive/MyDrive/hey_emma_training/{MODEL_NAME}'
LOCAL_PIPER_MODEL = f'/content/piper-sample-generator/models/{MODEL_NAME}'

os.makedirs('/content/piper-sample-generator/models', exist_ok=True)

if os.path.exists(DRIVE_PIPER_MODEL):
    print(f'Piper model found on Drive, copying...')
    !cp "$DRIVE_PIPER_MODEL" "$LOCAL_PIPER_MODEL"
elif os.path.exists(LOCAL_PIPER_MODEL):
    print(f'Piper model already local.')
else:
    print('Downloading Piper model...')
    !wget -q -O "$LOCAL_PIPER_MODEL" \
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
    # Cache to Drive
    !cp "$LOCAL_PIPER_MODEL" "$DRIVE_PIPER_MODEL"
    print('Piper model cached to Drive.')

PIPER_PATH = '/content/piper-sample-generator'
print(f'Piper path: {PIPER_PATH}')
print(f'Model exists: {os.path.exists(LOCAL_PIPER_MODEL)}')

## 2. Download Training Data

Alle Downloads werden auf Google Drive gespeichert (2 TB verfügbar).
Bereits vorhandene Dateien werden übersprungen.
Große Dateien werden erst lokal heruntergeladen, dann nach Drive kopiert.

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

DRIVE_DATA = '/content/drive/MyDrive/hey_emma_training/training_data'

for filename in [
    'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    'validation_set_features.npy',
]:
    dest = f'/content/training_data/{filename}'
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / (1024**2)
        print(f'{filename}: already on Drive ({size_mb:.0f} MB)')
    else:
        print(f'Downloading {filename}...')
        # Download to local HF cache first, then copy to Drive (avoids FUSE issues)
        local_path = hf_hub_download(
            repo_id='davidscripka/openwakeword_features',
            filename=filename,
            repo_type='dataset',
        )
        print(f'  Copying to Drive...')
        shutil.copy2(local_path, dest)
        size_mb = os.path.getsize(dest) / (1024**2)
        print(f'  Done ({size_mb:.0f} MB)')

In [ ]:
import os

rir_dir = '/content/training_data/mit_rirs/RIRS_NOISES'
musan_dir = '/content/training_data/musan'

# RIRs: extract locally first, then copy to Drive
if os.path.exists(rir_dir):
    print('MIT RIRs already on Drive.')
else:
    print('Downloading MIT RIRs...')
    !wget -q https://www.openslr.org/resources/28/rirs_noises.zip -O /tmp/rirs_noises.zip
    !unzip -q -o /tmp/rirs_noises.zip -d /tmp/mit_rirs
    print('Copying to Drive...')
    !cp -r /tmp/mit_rirs /content/training_data/mit_rirs
    !rm -rf /tmp/rirs_noises.zip /tmp/mit_rirs
    print('Done.')

# MUSAN: extract locally first, then copy to Drive (many small files)
if os.path.exists(musan_dir):
    print('MUSAN already on Drive.')
else:
    print('Downloading MUSAN...')
    !wget -q https://www.openslr.org/resources/17/musan.tar.gz -O /tmp/musan.tar.gz
    !tar -xzf /tmp/musan.tar.gz -C /tmp
    print('Copying to Drive (many small files, may take a moment)...')
    !cp -r /tmp/musan /content/training_data/musan
    !rm -rf /tmp/musan.tar.gz /tmp/musan
    print('Done.')

## 3. Training Config

Die Config wird direkt hier erstellt — keine externe Datei nötig.

In [ ]:
import yaml, os

PIPER_PATH = '/content/piper-sample-generator'

config = {
    'model_name': 'hey_emma',
    'target_phrase': ['hey emma'],
    'custom_negative_phrases': [
        'hey anna', 'hey ella', 'hey eva', 'hey ever',
        'hey oma', 'hey irma', 'hey mama', 'hey lemma', 'hey thema',
    ],
    'n_samples': 50000,
    'n_samples_val': 5000,
    'tts_batch_size': 100,
    'piper_sample_generator_path': PIPER_PATH,
    'output_dir': './hey_emma_model',
    'augmentation_batch_size': 16,
    'augmentation_rounds': 2,
    'rir_paths': ['./training_data/mit_rirs/RIRS_NOISES/simulated_rirs'],
    'background_paths': ['./training_data/musan/noise'],
    'background_paths_duplication_rate': [1],
    'feature_data_files': {
        'ACAV100M_sample': './training_data/openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    },
    'false_positive_validation_data_path': './training_data/validation_set_features.npy',
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('hey_emma.yml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('Config written to hey_emma.yml')

## 4. Write Training Wrapper

Compatibility shims for Colab (torchaudio API change + PyTorch 2.6 weights_only default).

In [ ]:
%%writefile run_train.py
# --- Compatibility shims for Colab ---
import sys, os

# 1. torchaudio >= 2.2 removed list_audio_backends()
import torchaudio
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile', 'sox']

# 2. PyTorch 2.6 changed torch.load default to weights_only=True
#    Piper's model needs the old behavior (trusted local model file)
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# 3. Add piper-sample-generator to sys.path BEFORE importing train.py
PIPER_PATH = '/content/piper-sample-generator'
if PIPER_PATH not in sys.path:
    sys.path.insert(0, PIPER_PATH)

# Verify generate_samples is importable
try:
    from generate_samples import generate_samples
    print(f'generate_samples imported from {PIPER_PATH}')
except ImportError:
    print(f'ERROR: generate_samples.py not found in {PIPER_PATH}')
    print(f'Contents: {os.listdir(PIPER_PATH) if os.path.exists(PIPER_PATH) else "DIR NOT FOUND"}')
    sys.exit(1)

# 4. Download melspectrogram.onnx and embedding model if missing
import openwakeword
openwakeword.utils.download_models()

# --- Run training script ---
import runpy

TRAIN_SCRIPT = os.path.join(os.path.dirname(openwakeword.__file__), 'train.py')
print(f'train.py: {TRAIN_SCRIPT}')

phase = sys.argv[1] if len(sys.argv) > 1 else '--generate_clips'
sys.argv = [TRAIN_SCRIPT, '--training_config', 'hey_emma.yml', phase]
runpy.run_path(TRAIN_SCRIPT, run_name='__main__')

## 5. Generate Synthetic Clips

Piper TTS erzeugt 50.000 Varianten von "Hey Emma" mit ~904 verschiedenen Stimmen.

In [ ]:
!python run_train.py --generate_clips

## 6. Augment Clips

Wendet Room Impulse Responses und Hintergrundgeräusche an.
2 Augmentation-Runden verdoppeln die Datenvielfalt.

In [ ]:
!python run_train.py --augment_clips

## 7. Extract Features

Extrahiert Audio-Embeddings aus den generierten Clips.
`AudioFeatures` speichert die berechneten Features in `feature_buffer`.

In [ ]:
import os, glob
import numpy as np
import soundfile as sf
from openwakeword.utils import AudioFeatures

base_dir = '/content/hey_emma_model/hey_emma'

for prefix, folders in [('positive', ['positive_train', 'positive_val']),
                        ('negative', ['negative_train', 'negative_val'])]:
    for folder in folders:
        clip_dir = f'{base_dir}/{folder}'
        if not os.path.exists(clip_dir):
            print(f'Skip {folder}')
            continue
        clips = sorted(glob.glob(f'{clip_dir}/*.wav'))
        print(f'{folder}: {len(clips)} clips')

        all_features = []
        for i, clip in enumerate(clips):
            af = AudioFeatures(device='cpu')
            audio, sr = sf.read(clip)
            audio_int16 = (audio * 32767).astype(np.int16)
            af(audio_int16)
            if af.feature_buffer.shape[0] > 0:
                all_features.append(af.feature_buffer.copy())
            if (i + 1) % 5000 == 0:
                print(f'  {i + 1}/{len(clips)}')

        suffix = 'val' if 'val' in folder else 'train'
        out = f'{base_dir}/{prefix}_features_{suffix}.npy'
        if all_features:
            stacked = np.concatenate(all_features, axis=0)
            np.save(out, stacked)
            print(f'Saved {out}: {len(all_features)} clips, shape {stacked.shape}')
        else:
            print(f'WARNING: No features for {folder}')

print('Feature extraction done!')

## 8. Train Model

Trainiert ein kleines DNN (2x 32 Units) auf den Audio-Embeddings.
Ziel: ≤ 0.2 False Positives pro Stunde.

In [ ]:
!python run_train.py --train_model

## 9. Export to ONNX + TFLite

In [ ]:
!python run_train.py --convert_to_tflite

In [ ]:
import glob, os

models = glob.glob('hey_emma_model/**/*.onnx', recursive=True)
models += glob.glob('hey_emma_model/**/*.tflite', recursive=True)

print('Exportierte Modelle:')
for m in sorted(models):
    size_kb = os.path.getsize(m) / 1024
    print(f'  {m} ({size_kb:.0f} KB)')

## 10. Quick Test

Schnelltest mit synthetischem Audio, um zu prüfen ob das Modell reagiert.

In [ ]:
import numpy as np, glob, wave
from openwakeword.model import Model

onnx_models = glob.glob('hey_emma_model/**/*.onnx', recursive=True)
model_path = [m for m in onnx_models if 'hey_emma' in m and 'embedding' not in m and 'melspec' not in m]

if model_path:
    print(f'Testing model: {model_path[0]}')
    oww = Model(wakeword_models=[model_path[0]], inference_framework='onnx')
    print(f'Loaded models: {list(oww.models.keys())}')

    val_clips = glob.glob('hey_emma_model/**/positive_val/*.wav', recursive=True)
    if val_clips:
        with wave.open(val_clips[0], 'rb') as wf:
            audio = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16)

        max_score = 0.0
        for i in range(0, len(audio) - 1280, 1280):
            chunk = audio[i:i+1280]
            prediction = oww.predict(chunk)
            for name, score in prediction.items():
                max_score = max(max_score, score)

        print(f'Max score on positive clip: {max_score:.3f}')
        print('PASS' if max_score > 0.5 else 'WARN: score below threshold, consider retraining')
    else:
        print('No validation clips found for testing.')
else:
    print('ERROR: No ONNX model found in output directory.')

## 11. Download Model

Das fertige Modell herunterladen und in `resources/` des Hey Emma Projekts ablegen.

In `.env` setzen:
```
OPENWAKEWORD_MODEL_PATH=hey_emma.onnx
OPENWAKEWORD_KEYWORD=Hey-Emma
```

In [ ]:
try:
    from google.colab import files
    if model_path:
        files.download(model_path[0])
        print('Download gestartet.')
except ImportError:
    if model_path:
        print(f'Modell bereit: {model_path[0]}')
        print('Kopiere die Datei nach resources/hey_emma.onnx im Projekt.')